# Lekcia 12 - Redukcia histórie rozhovoru pomocou Agent Scratchpad

Tento notebook demonštruje, ako spravovať kontext v dlhých rozhovoroch pomocou Microsoft Agent Framework. Ako rozhovory rastú, počet tokenov sa zvyšuje — až prekročí kontextové okno modelu. Tento problém riešime pomocou **vzoru sumarizácie kontextu** a **agent scratchpad** pre trvalú pamäť.

## Čo sa naučíte:
1. **Prečo je správa kontextu dôležitá**: Pochopenie limitov tokenov a kontextových okien
2. **Agenti vnímajúci kontext**: Budovanie agentov, ktorí si spravujú vlastný kontext rozhovoru
3. **Vzor sumarizácie kontextu**: Používanie nástrojov na zhrnutie histórie rozhovoru
4. **Agent Scratchpad**: Trvalá pamäť, ktorá pretrváva aj po redukcii kontextu

## Predpoklady:
- Nastavenie Azure OpenAI s nakonfigurovanými premennými prostredia
- Pochopenie základných konceptov agentov z predchádzajúcich lekcií


## Nastavenie


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## Prečo je správa kontextu dôležitá

Každý LLM má konečné **kontextové okno** — maximálny počet tokenov, ktoré môže spracovať v jednom požiadavku. Ako sa viackolová konverzácia vyvíja:

- **Počet tokenov rastie lineárne** s každou správou používateľa a odpoveďou asistenta.
- **Tokeny výzvy dominujú nákladom**, pretože celá história sa posiela znova pri každom kole.
- Nakoniec konverzácia **prekročí kontextové okno** a model buď orezáva text, alebo hlási chybu.

### Stratégie správy kontextu

| Stratégia | Ako funguje | Kompromis |
|---|---|---|
| **Orezanie** | Odstráni najstaršie správy | Stráca skorší kontext |
| **Zhrnutie** | Skondenzuje staršie správy do zhrnutia | Niektoré detaily sú stratené, ale kľúčové body zostávajú |
| **Scratchpad / externá pamäť** | Ukladá kľúčové fakty mimo konverzácie | Vyžaduje volania nástrojov, ale prežije akékoľvek zmenšenie |

V tomto zápisníku kombinujeme **zhrnutie** so **scratchpad nástrojom**, aby agent mohol udržať kontinuitu aj keď je história konverzácie skondenzovaná.


## Vytváranie kontextovo uvedomelého agenta


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## Simulácia dlhého rozhovoru

Prejdime si viackolový rozhovor, aby sme videli, ako sa kontext hromadí. Agent by mal zachovať kľúčové detaily (preferencie, rozpočet, dátumy cesty) naprieč kolami a preukázať kontinuitu.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Všimnite si, ako si agent uchováva kontext z predchádzajúcich kôl — vie o Japonsku, sushí, chrámoch, fotografovaní, rozpočte 3000 dolárov, sólo cestovaní a aprílovom časovom rámci. V krátkom rozhovore to funguje dobre, ale ako rozhovor rastie, celá história sa stáva nákladnou na opätovné odosielanie.

Pokračujme v rozhovore s viacerými kolami, aby sme videli akumuláciu kontextu:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Vzor súhrnu kontextu

Ako konverzácia rastie, môžeme použiť **nástroj na zhrnutie** na kondenzáciu nahromadeného kontextu do kompaktnej podoby. Agent zavolá tento nástroj, aby zaznamenal kľúčové preferencie, takže aj keď sa staršie správy odstránia, základné informácie zostanú zachované.

Tento vzor je stavebným prvkom pre sofistikovanejšie znižovanie histórie:
1. Agent identifikuje kľúčové fakty z konverzácie
2. Zavolá nástroj na zhrnutie, aby ich uchoval
3. Staršie správy možno bezpečne odstrániť, pretože súhrn zachytáva podstatné informácie

Nižšie definujeme nástroj `summarize_preferences`, ktorý môže agent zavolať na zaznamenanie kompaktnej súhrnnej správy o tom, čo sa naučil.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Zhrnutie

V tejto lekcii ste sa naučili, ako spravovať kontext v dlhodobých rozhovoroch agentov pomocou Microsoft Agent Framework:

### Kľúčové pojmy
- **Kontextové okná sú obmedzené** — každý token v histórii rozhovoru stojí peniaze a počíta sa do limitu.
- **Nástroje na zhrnutie** umožňujú agentovi zhrnúť nahromadený kontext do kompaktných súhrnov, čím sa znižuje používanie tokenov pri zachovaní podstatných informácií.
- **Poznámkové bloky agenta** poskytujú trvalú externú pamäť, ktorá pretrváva aj pri akomkoľvek znižovaní rozhovoru.

### Čo ste vytvorili
- **Kontextovo uvedomelého agenta**, ktorý udržuje kontinuitu v rámci viackolových rozhovorov
- **Nástroj na zhrnutie** (`summarize_preferences`), ktorý zaznamenáva dôležité používateľské detaily v kompaktnom formáte
- **Viackolový rozhovor** demonštrujúci uchovávanie kontextu a spracovanie zmien

### Skutočné použitia
- **Boti zákazníckej podpory**: Pamätajú si preferencie počas dlhých podporných sedení
- **Osobní asistenti**: Sledujú prebiehajúce projekty bez potreby opakovania kontextu
- **Vzdelávací lektori**: Udržiavajú pokrok študenta naprieč mnohými interakciami

### Ďalšie kroky
- Implementovať plnohodnotný poznámkový blok s perzistenciou založenou na súboroch
- Pridať automatické skracovanie histórie po zhrnutí
- Kombinovať s vektorovými databázami pre semantické vyhľadávanie v pamäti
- Vytvárať agentov, ktorí môžu o niekoľko dní pokračovať v rozhovoroch s úplným kontextom


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Upozornenie**:  
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Aj keď sa snažíme o presnosť, majte na pamäti, že automatické preklady môžu obsahovať chyby alebo nepresnosti. Pôvodný dokument v jeho rodnom jazyku by mal byť považovaný za autoritatívny zdroj. Pre kritické informácie sa odporúča profesionálny ľudský preklad. Nie sme zodpovední za akékoľvek nepochopenia alebo nesprávne interpretácie vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
